# Peak locations and fitted profiles

Runs the repaired `PeakyFinder` full path on a real broadband spectrum
(`data/remote_samples`, 180-961 nm) and shows:

1. **Found peak locations** over the background-subtracted spectrum, in the
   style of `finder.fit_spectrum_data()`'s diagnostic plot.
2. **Fitted Voigt profiles for individual peaks**, in the style of
   *3a-ii. Peak-shape PCs by segment* from `peaky_data.ipynb`: a panel grid
   with the local data, this peak's fitted component, and the rest of the
   model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import voigt_profile

from alibz import PeakyFinder

SPECTRUM = 0  # 0=REE_01, 1=REE_44, 2=argon_noAr, 3=scan9x9

finder = PeakyFinder('../data/remote_samples')
fit_dict = finder.fit_spectrum_data(SPECTRUM, n_sigma=0, subtract_background=True, plot=False)

x, y = finder.data.get_data()[SPECTRUM]
y_bgsub = y - fit_dict['background']
peaks = fit_dict['sorted_parameter_array']   # [area, center, sigma, gamma], strongest first
profile = fit_dict['profile']

fwhm = finder.voigt_width(np.maximum(peaks[:, 2], 1e-6), np.maximum(peaks[:, 3], 1e-6))
line_like = fwhm < 2.0  # exclude ultra-broad baseline-remnant components from per-peak display
print(f'{peaks.shape[0]} peaks kept ({int(line_like.sum())} line-like, '
      f'{int((~line_like).sum())} broad baseline components)')

## 1. Found peak locations

Style of `fit_spectrum_data()`: background-subtracted data in black, the
total fitted multi-Voigt profile filled in red, and a red marker at every
kept peak location (drawn at the height each fitted area implies).

In [ ]:
peak_heights = peaks[:, 0] * voigt_profile(
    0.0, np.maximum(peaks[:, 2], 1e-6), np.maximum(peaks[:, 3], 1e-6))

plt.rcParams.update({'font.size': 14})
fig, ax = plt.subplots(figsize=(24, 6))
ax.plot(x, y_bgsub, color='k', alpha=0.9, lw=0.7, label='background-subtracted data')
ax.plot(x, profile, color='r', lw=0.5)
ax.fill_between(x, profile, np.zeros_like(x), color='r', alpha=0.5,
                label='fitted multi-Voigt profile')
ax.scatter(peaks[line_like, 1], peak_heights[line_like], color='r', s=14,
           alpha=0.7, zorder=3, label=f'{int(line_like.sum())} peak locations')
ax.set_xlabel('wavelength [nm]')
ax.set_ylabel('intensity [counts]')
ax.set_xlim(x.min(), x.max())
ax.legend(loc='upper right')
plt.show()

# zoomable variant: log-y to reveal the weak-line forest
fig, ax = plt.subplots(figsize=(24, 4))
ax.semilogy(x, np.clip(y_bgsub, 0.1, None), color='k', alpha=0.9, lw=0.7)
ax.scatter(peaks[line_like, 1], np.clip(peak_heights[line_like], 0.1, None),
           color='r', s=10, alpha=0.7, zorder=3)
ax.set_xlabel('wavelength [nm]')
ax.set_ylabel('intensity [counts]')
ax.set_xlim(x.min(), x.max())
ax.set_title('log scale', fontweight='bold')
plt.show()

## 2. Fitted peak profiles for individual peaks

Style of *3a-ii*: one panel per peak (4 columns), each showing the local
data window (black), this peak's fitted Voigt component (coloured, filled),
and the summed contribution of all other components (grey dashed) — so
blends and wing pedestals are visible at a glance.  Shown: the 12 strongest
line-like peaks.

In [ ]:
n_show, n_cols = 12, 4
show_idx = np.flatnonzero(line_like)[:n_show]
n_rows = (len(show_idx) + n_cols - 1) // n_cols

pc_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b',
             '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#1f77b4', '#ff7f0e']

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

for panel, pi in enumerate(show_idx):
    amp, mu, sig, gam = peaks[pi]
    ax = axes[panel]
    half_span = max(6 * fwhm[pi], 0.35)
    m = (x >= mu - half_span) & (x <= mu + half_span)
    own = amp * voigt_profile(x[m] - mu, max(sig, 1e-6), max(gam, 1e-6))
    rest = profile[m] - own
    color = pc_colors[panel % len(pc_colors)]
    ax.plot(x[m], y_bgsub[m], 'k-', lw=1.2, label='data')
    ax.plot(x[m], own, color=color, lw=1.5, label='this peak')
    ax.fill_between(x[m], own, alpha=0.25, color=color)
    ax.plot(x[m], rest, color='0.5', lw=1.0, ls='--', alpha=0.8, label='other peaks')
    ax.set_title(f'{mu:.2f} nm   A={amp:.0f}  $\\sigma$={sig:.3f}  $\\gamma$={gam:.3f}',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('wavelength [nm]')
    ax.set_ylabel('intensity [counts]')
    ax.legend(fontsize=7)

for ax in axes[len(show_idx):]:
    ax.set_visible(False)

plt.suptitle('Fitted Voigt profiles — 12 strongest line-like peaks',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

To inspect specific peaks instead of the strongest ones, replace
`show_idx` above — e.g. peaks nearest a wavelength of interest:

```python
targets = [393.37, 396.85, 589.0]  # nm
show_idx = [int(np.argmin(np.abs(peaks[:, 1] - t))) for t in targets]
```